# Uncertainty Engine SDK - saving and loading workflows

This notebook goes through some of the basic functionality of the SDK. It demonstrates how to save and load your workflows.


## The `Client`

Start by importing the `Client`.


In [1]:
from uncertainty_engine import Client

Next initialise the `Client`.

In [ ]:
client = Client()

Before accessing your workflows you must authenticate and select the project you wish to save and load from.

Authentication is carried out with the `client.authenticate` method. The IDs for your available projects can be accessed by executing the `client.projects.list_projects` method. This will return a list of your available projects. Each with an `"id"` field.

In [ ]:
# Authenticate
client.authenticate()

# Select your project you wish to save and load from using your selected project ID
PROJECT_ID = "YOUR_PROJECT_ID"

## Saving a new workflow

Firstly, we will create a basic add workflow to save. 

In [4]:
from uncertainty_engine.graph import Graph

graph = Graph()

add = client.nodes.Add(rhs=1, lhs=42, label="Add")

# Add the node to the graph
graph.add_node(add)

# Define our add output handle
ans = add.make_handle("ans")

Make sure to wrap your basic workflow in the `Workflow` node before saving. The save method requires the workflow you wish to save to be an instance of the `Workflow` class.

In [5]:
from uncertainty_engine.nodes.workflow import Workflow
from pprint import pprint

basic_workflow = Workflow(
    graph=graph.nodes,
    inputs=graph.external_input,
    external_input_id=graph.external_input_id,
    requested_output={"Add": ans.model_dump()},
)

Now we can save our workflow. As it is a new workflow you will need to make sure you pass it a `workflow_name`. This must be unique to your other saved workflows.

You can tell your workflow has been saved successfully if your workflow ID is returned.

In [7]:
basic_workflow_id = client.workflows.save(
    project_id=PROJECT_ID,
    workflow=basic_workflow,
    workflow_name="MyBasicAddWorkflow",
)

print(basic_workflow_id)

6abe5d2074cea6705e9dcc8f


## Listing your existing workflows

We can now list our workflows using the `list_workflows` method to see our saved workflow details.

In [8]:
pprint([wf.model_dump() for wf in client.workflows.list_workflows(PROJECT_ID)])

[{'created_at': '09:39:07 2025-11-25',
  'id': '6925793b4f296b9e255966a6',
  'name': 'test-new-download',
  'owner_id': '686cf4b1232f44e48f0edceb',
  'versions': ['6925793b4f296b9e255966a7',
               '692579414f296b9e255966a8',
               '692579964f296b9e255966a9']},
 {'created_at': '15:57:49 2025-08-28',
  'id': '69286d1f7184b45cf9d548db',
  'name': 'workflow-no-versions',
  'owner_id': '686cf4b1232f44e48f0edceb',
  'versions': []},
 {'created_at': '11:09:15 2025-12-09',
  'id': '6938035b83abfefaf77c1b2c',
  'name': 'torch-script-test',
  'owner_id': '686cf4b1232f44e48f0edceb',
  'versions': ['6938035c83abfefaf77c1b2d',
               '6938037483abfefaf77c1b2e',
               '693803c383abfefaf77c1b35']},
 {'created_at': '12:02:18 2025-12-09',
  'id': '69380fca83abfefaf77c1b6a',
  'name': 'train-export-torch',
  'owner_id': '686cf4b1232f44e48f0edceb',
  'versions': ['69380fca83abfefaf77c1b6b',
               '693810e983abfefaf77c1b73',
               '693811a483abfefaf77c1

You will see that along side the ID this method returns the name, list of versions and time the workflow was created. 

This is a particularly useful method when you start having many workflows and would like to view which one you wish to load.

## Loading your saved workflow

When loading a resource you must provide the `project_id` and the `workflow_id`. Optionally, if you wish to load a specific previous version of your workflow, you can provide the `version_id` as well. Otherwise, the latest version of your workflow will be loaded.

In [9]:
loaded_workflow = client.workflows.load(
    project_id=PROJECT_ID, workflow_id=basic_workflow_id
)

Now we have loaded our basic add workflow we can run it as shown below.

In [10]:
response = client.run_node(loaded_workflow)
pprint(response.model_dump())

{'inputs': {'_nodes_list_error': None,
            'external_input_id': '_',
            'graph': {'nodes': {'Add': {'inputs': {'lhs': {'node_handle': 'Add_lhs',
                                                           'node_name': '_'},
                                                   'rhs': {'node_handle': 'Add_rhs',
                                                           'node_name': '_'}},
                                        'type': 'Add',
                                        'version': 'latest'}}},
            'inputs': {'Add_lhs': 42, 'Add_rhs': 1},
            'requested_output': {'Add': {'node_handle': 'ans',
                                         'node_name': 'Add'}}},
 'message': 'Job completed at 2026-10-01 13:16:53.037212',
 'outputs': {'outputs': {'Add': 43.0}},
 'progress': {'Add': {'inputs': {'lhs': 42, 'rhs': 1},
                      'message': 'Job completed at 2026-10-01 13:16:52.579567',
                      'outputs': {'ans': 43.0},
               

## Creating a new version of an existing workflow

When we wish to save a new version of an already existing workflow we can simply use the `save` method but make sure to provide the `workflow_id` under which you wish the new version to be saved.

Note that when saving an already existing workflow, the `workflow_name` is not required.

In [11]:
client.workflows.save(
    project_id=PROJECT_ID,
    workflow=basic_workflow,
    workflow_id=basic_workflow_id,
)

# List all workflows in the project
pprint([wf.model_dump() for wf in client.workflows.list_workflows(PROJECT_ID)])

[{'created_at': '09:39:07 2025-11-25',
  'id': '6925793b4f296b9e255966a6',
  'name': 'test-new-download',
  'owner_id': '686cf4b1232f44e48f0edceb',
  'versions': ['6925793b4f296b9e255966a7',
               '692579414f296b9e255966a8',
               '692579964f296b9e255966a9']},
 {'created_at': '15:57:49 2025-08-28',
  'id': '69286d1f7184b45cf9d548db',
  'name': 'workflow-no-versions',
  'owner_id': '686cf4b1232f44e48f0edceb',
  'versions': []},
 {'created_at': '11:09:15 2025-12-09',
  'id': '6938035b83abfefaf77c1b2c',
  'name': 'torch-script-test',
  'owner_id': '686cf4b1232f44e48f0edceb',
  'versions': ['6938035c83abfefaf77c1b2d',
               '6938037483abfefaf77c1b2e',
               '693803c383abfefaf77c1b35']},
 {'created_at': '12:02:18 2025-12-09',
  'id': '69380fca83abfefaf77c1b6a',
  'name': 'train-export-torch',
  'owner_id': '686cf4b1232f44e48f0edceb',
  'versions': ['69380fca83abfefaf77c1b6b',
               '693810e983abfefaf77c1b73',
               '693811a483abfefaf77c1

As you can see above `MyBasicAddWorkflow` now has another version in its list of versions.